In [1]:
import sys; sys.path.append('../3rdparty/ElasticKnots/3rdparty/ElasticRods/python')
import sys; sys.path.append('../3rdparty/ElasticKnots/python')
import elastic_rods, elastic_knots
import numpy as np, matplotlib.pyplot as plt, time, io, os
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import eigsh
from scipy.linalg import eigh

from helpers import *
from parametric_curves import *
import py_newton_optimizer 

from linkage_vis import LinkageViewer as Viewer, CenterlineViewer
from tri_mesh_viewer import PointCloudViewer, PointCloudMesh

%load_ext autoreload
%autoreload 2

import parallelism
parallelism.set_max_num_tbb_threads(1)

from MEP import MEP

Failed to load offscreen viewer: Could not load compiled module; is OffscreenRenderer missing a dependency?


In [6]:
knot_name = '6_3/0010.obj'
file = '../data/L400-r0.2-UpTo9Crossings/' + knot_name
rod_radius = 0.2
material = elastic_rods.RodMaterial('ellipse', 2000, 1, [rod_radius, rod_radius])
centerline = read_nodes_from_file(file)  # supported formats: obj, txt
pr = define_periodic_rod(centerline[::], material)
rod_list = elastic_knots.PeriodicRodList([pr])
len(rod_list.getDoFs())

1601

In [7]:
view = Viewer(rod_list, width=1024, height=800)
view.show()


Renderer(camera=PerspectiveCamera(aspect=1.28, children=(PointLight(color='#999999', position=(0.0, 0.0, 5.0),…

In [8]:
def callback(problem, iteration):
    if iteration % 5 == 0:
        view.update()
for i in range(10,30):
    print(f"iterration: {i}")
    optimizerOptions = py_newton_optimizer.NewtonOptimizerOptions()
    optimizerOptions.niter = 1000
    optimizerOptions.gradTol = 1e-6
    hessianShift = 1e-4 * compute_min_eigenval_straight_rod(pr)

    problemOptions = elastic_knots.ContactProblemOptions()
    problemOptions.contactStiffness = 1e+3
    problemOptions.dHat = 2*rod_radius * 0.1*i
    fixedVars = []   
    
    report = elastic_knots.compute_equilibrium(
        rod_list, problemOptions, optimizerOptions, 
        fixedVars=fixedVars,
        externalForces=np.zeros(rod_list.numDoF()),
        softConstraints=[],
        callback=callback,
        hessianShift=hessianShift
        )
    view.update()

iterration: 10
0	1.47298	0.829166	0.829166	1	1
1	1.38178	0.00623762	0.00623762	1	0
2	1.38178	3.10006e-05	3.10006e-05	1	0
3	1.38178	3.17026e-08	3.17026e-08	1	0
iterration: 11
0	4.61349	116.506	116.506	1	1
1	1.91668	32.993	32.993	1	1
2	1.48836	9.5406	9.5406	1	1
3	1.4134	2.87508	2.87508	1	1
4	1.39589	0.850982	0.850982	1	1
5	1.38987	0.295223	0.295223	1	1
6	1.38676	0.13268	0.13268	1	1
7	1.38497	0.0696589	0.0696589	1	1
8	1.38394	0.0355768	0.0355768	1	1
9	1.38338	0.0188013	0.0188013	1	1
10	1.38305	0.0110141	0.0110141	1	1
11	1.38281	0.00608999	0.00608999	1	1
12	1.38266	0.00351709	0.00351709	1	1
13	1.38257	0.00193488	0.00193488	1	1
14	1.38251	0.00105154	0.00105154	1	1
15	1.38248	0.00061596	0.00061596	1	1
16	1.38245	0.000395472	0.000395472	1	1
17	1.38243	0.000283751	0.000283751	1	1
18	1.38241	0.000266854	0.000266854	1	1
19	1.3824	0.000335326	0.000335326	1	1
20	1.3824	0.000593163	0.000593163	1	1
21	1.38239	0.00213837	0.00213837	1	1
22	1.38239	0.000458776	0.000458776	1	1
23	1.38238	0.000666962	0.0

The simulations will continue, but the result might be non-physical due to spurious contact forces between neighboring edges.
Consider decreasing the cross-section radius or using a coarser polyline.
Increasing the minContactEdgeDist parameter would remove non-physical contact forces, too, but only at the expense of topology preservation guarantees.



0	11.4335	388.35	388.35	1	1
1	2.77432	88.3138	88.3138	1	1
2	1.63317	22.9442	22.9442	1	1
3	1.45273	6.26375	6.26375	1	1
4	1.41112	1.66567	1.66567	1	1
5	1.3993	0.475983	0.475983	1	1
6	1.39487	0.154944	0.154944	1	1
7	1.39279	0.0663433	0.0663433	1	1
8	1.39168	0.0330835	0.0330835	1	1
9	1.39104	0.0175928	0.0175928	1	1
10	1.39067	0.00939527	0.00939527	1	1
11	1.39048	0.00491957	0.00491957	1	1
12	1.39038	0.00277263	0.00277263	1	1
13	1.39033	0.00137421	0.00137421	1	1
14	1.39029	0.000701059	0.000701059	1	1
15	1.39027	0.000435785	0.000435785	0.015625	0
16	1.39027	0.00389462	0.00389462	1	1
17	1.39025	0.000803396	0.000803396	0.125	0
18	1.39024	0.00998812	0.00998812	0.125	0
19	1.39024	0.017897	0.017897	1	0
20	1.39022	0.00654153	0.00654153	1	0
21	1.39021	0.00172897	0.00172897	1	1
22	1.39021	0.000379041	0.000379041	1	1
23	1.39021	2.61361e-05	2.61361e-05	1	1
24	1.39021	5.34761e-06	5.34761e-06	1	1
25	1.39021	7.29344e-06	7.29344e-06	1	1
26	1.39021	1.06568e-05	1.06568e-05	1	1
27	1.39021	1.15306e-05	1.15306e

The simulations will continue, but the result might be non-physical due to spurious contact forces between neighboring edges.
Consider decreasing the cross-section radius or using a coarser polyline.
Increasing the minContactEdgeDist parameter would remove non-physical contact forces, too, but only at the expense of topology preservation guarantees.



0	10.8462	364.606	364.606	1	1
1	2.78969	93.8484	93.8484	1	1
2	1.61862	25.0873	25.0873	1	1
3	1.43974	6.75776	6.75776	1	1
4	1.40681	1.83993	1.83993	1	1
5	1.39818	0.503264	0.503264	1	1
6	1.39493	0.154818	0.154818	1	1
7	1.39325	0.0639946	0.0639946	1	1
8	1.39228	0.0320474	0.0320474	1	1
9	1.39169	0.0174456	0.0174456	1	1
10	1.39134	0.00964021	0.00964021	1	1
11	1.39115	0.0054719	0.0054719	1	1
12	1.39105	0.00251912	0.00251912	1	1
13	1.39099	0.00128072	0.00128072	1	1
14	1.39096	0.000734005	0.000734005	1	1
15	1.39094	0.000470812	0.000470812	1	1
16	1.39092	0.000287732	0.000287732	1	1
17	1.39091	0.00620953	0.00620953	1	1
18	1.3909	0.00136172	0.00136172	1	1
19	1.3909	0.00029213	0.00029213	1	1
20	1.3909	0.000336236	0.000336236	1	1
21	1.39089	0.000231593	0.000231593	1	1
22	1.39089	0.0103856	0.0103856	1	1
23	1.39089	0.00198842	0.00198842	0.5	0
24	1.39089	0.00348496	0.00348496	0.25	0
25	1.39088	0.0022219	0.0022219	1	0
26	1.39088	0.00109728	0.00109728	0.125	0
27	1.39088	0.00104181	0.00104181	1	0
28	1.390

...

0	10.9527	374.818	374.818	1	1
1	2.75825	95.7665	95.7665	1	1
2	1.60182	25.0964	25.0964	1	1
3	1.4354	6.66122	6.66122	1	1
4	1.4067	1.8573	1.8573	1	1
5	1.39902	0.493719	0.493719	1	1
6	1.39582	0.165633	0.165633	1	1
7	1.39403	0.0671658	0.0671658	1	1
8	1.39301	0.0332945	0.0332945	1	1
9	1.39239	0.0181774	0.0181774	1	1
10	1.39201	0.00998386	0.00998386	1	1
11	1.39181	0.0056498	0.0056498	1	1
12	1.39171	0.00320609	0.00320609	1	1
13	1.39165	0.00135142	0.00135142	1	1
14	1.39162	0.000732605	0.000732605	1	1
15	1.3916	0.000446971	0.000446971	1	1
16	1.39158	0.000284843	0.000284843	1	1
17	1.39157	0.000195123	0.000195123	1	1
18	1.39156	0.00019964	0.00019964	1	1
19	1.39156	0.000247781	0.000247781	1	1
20	1.39155	0.00124131	0.00124131	1	1
21	1.39155	0.000104822	0.000104822	0.5	0
22	1.39155	0.011781	0.011781	0.03125	1
23	1.39155	0.0124797	0.0124797	1	1
24	1.39154	0.00659444	0.00659444	1	1
25	1.39154	0.00108336	0.00108336	0.25	0
26	1.39154	0.00249846	0.00249846	0.5	0
27	1.39154	0.00127859	0.00127859	1	0
28	1.3

The simulations will continue, but the result might be non-physical due to spurious contact forces between neighboring edges.
Consider decreasing the cross-section radius or using a coarser polyline.
Increasing the minContactEdgeDist parameter would remove non-physical contact forces, too, but only at the expense of topology preservation guarantees.



0	10.8194	356.427	356.427	1	1
1	2.74282	92.5979	92.5979	1	1
2	1.60466	24.5467	24.5467	1	1
3	1.43456	6.45248	6.45248	1	1
4	1.40647	1.68711	1.68711	1	1
5	1.39941	0.456849	0.456849	1	1
6	1.39637	0.152129	0.152129	1	1
7	1.3947	0.0653836	0.0653836	1	1
8	1.39372	0.0343574	0.0343574	1	1
9	1.3931	0.019438	0.019438	1	1
10	1.39272	0.0105434	0.0105434	1	1
11	1.39252	0.00587668	0.00587668	1	1
12	1.39241	0.00350194	0.00350194	1	1
13	1.39235	0.00142135	0.00142135	1	1
14	1.39232	0.000755716	0.000755716	1	1
15	1.3923	0.000457978	0.000457978	1	1
16	1.39228	0.000290596	0.000290596	1	1
17	1.39227	0.000196542	0.000196542	1	1
18	1.39226	0.000187119	0.000187119	1	1
19	1.39225	0.000226033	0.000226033	1	1
20	1.39225	0.000197945	0.000197945	1	1
21	1.39225	0.00570565	0.00570565	1	1
22	1.39224	0.000972844	0.000972844	0.125	0
23	1.39224	0.00228339	0.00228339	0.25	0
24	1.39224	0.0051472	0.0051472	0.000244141	1
25	1.39224	0.00686418	0.00686418	1	1
26	1.39224	0.00587568	0.00587568	1	1
27	1.39224	0.00137939	0.0013793

The simulations will continue, but the result might be non-physical due to spurious contact forces between neighboring edges.
Consider decreasing the cross-section radius or using a coarser polyline.
Increasing the minContactEdgeDist parameter would remove non-physical contact forces, too, but only at the expense of topology preservation guarantees.



0	12.0089	413.801	413.801	1	1
1	2.94087	107.089	107.089	1	1
2	1.64213	28.4324	28.4324	1	1
3	1.44237	7.5269	7.5269	1	1
4	1.40854	2.05062	2.05062	1	1
5	1.40037	0.572883	0.572883	1	1
6	1.39715	0.168568	0.168568	1	1
7	1.39545	0.0684229	0.0684229	1	1
8	1.39444	0.034702	0.034702	1	1
9	1.39381	0.0190815	0.0190815	1	1
10	1.39343	0.01054	0.01054	0.5	0
11	1.39334	0.265563	0.265563	0.25	1
12	1.39327	0.228096	0.228096	1	1
13	1.39303	0.0563444	0.0563444	1	1
14	1.39299	0.0133682	0.0133682	1	1
15	1.39297	0.00274538	0.00274538	1	1
16	1.39296	0.000376927	0.000376927	0.000732422	0
17	1.39296	0.00742881	0.00742881	1	1
18	1.39296	0.00121018	0.00121018	1	1
19	1.39295	0.000310132	0.000310132	1	1
20	1.39295	0.000327243	0.000327243	1	1
21	1.39295	0.000411438	0.000411438	1	1
22	1.39294	0.000159666	0.000159666	1	1
23	1.39294	0.000298487	0.000298487	1	1
24	1.39293	0.000738243	0.000738243	1	1
25	1.39293	0.00101296	0.00101296	0.5	1
26	1.39293	0.00367936	0.00367936	1	1
27	1.39292	0.000841938	0.000841938	0.03125	0
2

In [9]:
from helpers import write_obj
file = '../data/NoCollision/' + knot_name
write_obj(file, rod_list)

In [10]:
# Load the centerline from file...
file = '../data/NoCollision/' + knot_name
knot = read_nodes_from_file(file)
rod_radius = 0.2
material = elastic_rods.RodMaterial('ellipse', 2000, 0.3, [rod_radius, rod_radius])
pr = define_periodic_rod(knot[::4], material)
rod_list = elastic_knots.PeriodicRodList([pr])

In [11]:
view = Viewer(rod_list, width=1024, height=800)
view.show()

Renderer(camera=PerspectiveCamera(aspect=1.28, children=(PointLight(color='#999999', position=(0.0, 0.0, 5.0),…

In [12]:
from helpers import write_obj
file = '../data/NoCollision/reduced' + knot_name
write_obj(file, rod_list)